# BERT をファインチューニングして文書分類する (Trainer)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/13_classification/bert_finetune.ipynb)

NLP クックブック 13-09 ／ テキスト分類

> GPU ランタイム推奨 (ランタイム → ランタイムのタイプを変更 → T4 GPU)

事前学習済みの日本語 BERT を自分の分類データでファインチューニングして、高精度な分類器を作りたい。

In [ ]:
!pip install -q fugashi unidic-lite datasets evaluate accelerate

In [ ]:
import io, tarfile, urllib.request
import pandas as pd
from datasets import ClassLabel, Dataset

def load_livedoor():
    url = "https://www.rondhuit.com/download/ldcc-20140209.tar.gz"
    with urllib.request.urlopen(url) as r:
        tar = tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz")
    rows = []
    for m in tar.getmembers():
        parts = m.name.split("/")
        if m.isfile() and len(parts) == 3 and parts[2] != "LICENSE.txt":
            lines = tar.extractfile(m).read().decode("utf-8").split("\n")
            rows.append({"label": parts[1], "title": lines[2], "text": "\n".join(lines[3:])})
    return pd.DataFrame(rows)

df = load_livedoor()
names = sorted(df["label"].unique())
ds = Dataset.from_pandas(df).cast_column("label", ClassLabel(names=names))   # 文字列ラベル → 整数 ID
ds = ds.train_test_split(test_size=0.2, stratify_by_column="label", seed=0)
print(ds)

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding

model_name = "tohoku-nlp/bert-base-japanese-v3"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess(batch):
    return tokenizer(batch["title"], batch["text"], truncation="only_second", max_length=256)

tokenized = ds.map(preprocess, batched=True, remove_columns=["title", "text"])
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=len(names), id2label=dict(enumerate(names)), label2id={n: i for i, n in enumerate(names)})

In [ ]:
import evaluate
import numpy as np
from transformers import Trainer, TrainingArguments

accuracy, f1 = evaluate.load("accuracy"), evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred = np.argmax(logits, axis=-1)
    return {**accuracy.compute(predictions=pred, references=labels),
            **f1.compute(predictions=pred, references=labels, average="macro")}

args = TrainingArguments(
    output_dir="out", num_train_epochs=3, learning_rate=3e-5, warmup_steps=50, weight_decay=0.01,
    per_device_train_batch_size=32, per_device_eval_batch_size=64,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="f1",
    fp16=True, logging_steps=50, report_to="none", seed=0,
)
trainer = Trainer(model=model, args=args, train_dataset=tokenized["train"], eval_dataset=tokenized["test"],
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
trainer.train()
print(trainer.evaluate())

In [ ]:
# 保存して pipeline で推論
from transformers import pipeline

trainer.save_model("livedoor-bert"); tokenizer.save_pretrained("livedoor-bert")
clf = pipeline("text-classification", model="livedoor-bert", device=0)
print(clf("新型スマートフォンは有機 EL ディスプレイと高性能カメラを搭載し、来月発売される。"))

- T4 GPU で 1 エポック約 2 分。正解率は 95〜97% 程度になる。
- 学習率 (1e-5〜5e-5) とエポック数 (2〜5) が最も効くハイパーパラメータ。小さいデータではシードによるばらつきも大きいので、複数シードで平均をとる。
- 本来は train / dev / test の 3 分割にし、dev でモデル選択、test は最後に 1 回だけ使う。